NOTEBOOK INGESTION_SQL_SERVER

Responsável por mover os dados da GOLD para o SQL Server.

# CONFIGURAÇÕES GERAIS

## Rodar notebooks de configuração

In [0]:
%run ../../config/feat_squad2_config_adls

In [0]:
%run ../../config/feat_squad2_config_sqlserver

In [0]:
%run ../../utils/feat_squad2_utils

## Configurar as variáveis

In [0]:
entity_name = dbutils.widgets.get("entity_name")
folder_name_source = f"silver/ecommerce_{entity_name}"
path_source = f"abfs://{container_name_data_lake}/silver/ecommerce_{entity_name}"
path_gold_base = f"abfs://{container_name_data_lake}/gold"
check_interval = 10 
snapshot_id = 0

storage_options = {
    "storage_account_name": storage_account_name,
    "tenant_id": tenant_id,
    "client_id": client_id,
    "client_secret": client_secret
}

In [0]:
gold_tables = [
    {
        "path": f"abfs://{container_name_data_lake}/gold/new_customers",
        "table_name": "squad2.gold_ecommerce_new_customers",
        "mode": "append"
    },
    {
        "path": f"abfs://{container_name_data_lake}/gold/email_provider_rate",
        "table_name": "squad2.gold_ecommerce_email_provider_rate",
        "mode": "overwrite"
    },
    {
        "path": f"abfs://{container_name_data_lake}/gold/first_purchase",
        "table_name": "squad2.gold_ecommerce_first_purchase",
        "mode": "overwrite"
    },
    {
        "path": f"abfs://{container_name_data_lake}/gold/address_distribution_by_state",
        "table_name": "squad2.gold_ecommerce_address_distribution_by_state",
        "mode": "overwrite"
    }
]

In [0]:
for gold_table in gold_tables:

    log.info(f"Iniciando carga da Gold para SQL Server: {gold_table['table_name']}"
    )

    df_gold = read_delta_to_spark_df(
        path=gold_table["path"],
        storage_options=storage_options 
    )

    write_sql_server(
        df=df_gold,
        table_name=gold_table["table_name"],
        jdbc_hostname=jdbc_hostname,
        jdbc_database=jdbc_database,
        jdbc_username=jdbc_username,
        jdbc_password=jdbc_password,
        mode=gold_table["mode"]
    )

    log.info(f"Carga concluída: {gold_table['table_name']}")

TESTE

In [0]:
# Ler gold new customers

df_gold_new_customers = read_delta_to_spark_df(
    path= f"{path_gold_base}/new_customers",
    storage_options=storage_options
)

display(df_gold_new_customers)

In [0]:
# Ler gold email_provider_rate

df_gold_email_provider_rate = read_delta_to_spark_df(
    path= f"{path_gold_base}/email_provider_rate",
    storage_options=storage_options
)

display(df_gold_email_provider_rate)

In [0]:
# Ler gold address_distribution_by_state

df_gold_address_distribution_by_state = read_delta_to_spark_df(
    path= f"{path_gold_base}/address_distribution_by_state",
    storage_options=storage_options
)

display(df_gold_address_distribution_by_state)

In [0]:
# Ler gold first_purchase

df_gold_first_purchase = read_delta_to_spark_df(
    path=f"{path_gold_base}/first_purchase",
    storage_options=storage_options
)

display(df_gold_first_purchase)

In [0]:
# lista arquivos na Silver
all_files = list_files(
                    container_client = container_client_data_lake,
                    folder_name = "gold"
)                    
print(f"print files in container:")
for files in all_files:
    print(files)

In [0]:
# Limpar Gold
container_client_data_lake.delete_directory("gold/address_distribution_by_state")

In [0]:
# Limpar Gold
container_client_data_lake.delete_directory("gold/email_provider_rate")

In [0]:
# Limpar Gold
container_client_data_lake.delete_directory("gold/new_customers")

In [0]:
# Limpar Gold
container_client_data_lake.delete_directory("gold/first_purchase")